In [ ]:
%load_ext autoreload
%autoreload 2

import os
import numpy as np
import scvelo as scv
import scanpy as sc

# --------------------------
# 1. Output directory
# --------------------------
outdir = "./data/benchmark/velocity_estimation/dentate_gyrus"
os.makedirs(outdir, exist_ok=True)

# --------------------------
# 2. Load dataset (WITH existing UMAP)
# --------------------------
adata = scv.datasets.dentategyrus()
print("Loaded dentate gyrus:", adata)

# --------------------------
# 3. Save ORIGINAL scVelo UMAP
# --------------------------
X_umap = adata.obsm["X_umap"].copy()
np.save(f"{outdir}/embedding.npy", X_umap)
print("Saved ORIGINAL UMAP:", X_umap.shape)

# --------------------------
# 4. Define pseudotime exclusion mask
# --------------------------
exclude_types = {
    "Cajal-Retzius",
    "GABA-Lhx6",
    "GABA-Cnr1",
    "OPC",
    "NFOL",
    "OL",
    "Microglia",
    "PVM",
    "VLMC",
    "Endothelial",
    "Pericytes",
}

valid_type_mask = ~adata.obs["clusters_enlarged"].isin(exclude_types)

umap1 = X_umap[:, 0]
outlier_idx = np.argsort(umap1)[-3:]
outlier_mask = np.ones(adata.n_obs, dtype=bool)
outlier_mask[outlier_idx] = False

valid_pseudotime_mask = valid_type_mask & outlier_mask

print(f"Pseudotime-valid cells: {valid_pseudotime_mask.sum()} / {adata.n_obs}")

# --------------------------
# 5. Standard scVelo preprocessing
# --------------------------
scv.pp.filter_and_normalize(
    adata,
    min_shared_counts=30,
    n_top_genes=2000,
)
scv.pp.moments(adata, n_pcs=50, n_neighbors=30)

print("After preprocessing:", adata.shape)

# --------------------------
# 6. Save processed spliced/unspliced
#    (these are aligned with velocity)
# --------------------------
S = adata.layers["spliced"]
U = adata.layers["unspliced"]

if not isinstance(S, np.ndarray):
    S = S.toarray()
if not isinstance(U, np.ndarray):
    U = U.toarray()

np.save(f"{outdir}/spliced.npy", S)
np.save(f"{outdir}/unspliced.npy", U)
print("Saved processed S, U:", S.shape, U.shape)

# optional but useful for sanity/debugging
np.save(f"{outdir}/genes.npy", adata.var_names.to_numpy())

# --------------------------
# 7. Velocity computation (GENE SPACE)
# --------------------------
print("Computing stochastic velocity...")
scv.tl.velocity(adata, mode="stochastic", vkey="stochastic_velocity")
scv.tl.velocity_graph(adata, vkey="stochastic_velocity")

V_stoch = adata.layers["stochastic_velocity"]
if not isinstance(V_stoch, np.ndarray):
    V_stoch = V_stoch.toarray()

np.save(f"{outdir}/V_stochastic.npy", V_stoch)
print("Saved V_stochastic:", V_stoch.shape)

print("Computing dynamical velocity...")
scv.tl.recover_dynamics(adata)
scv.tl.velocity(adata, mode="dynamical", vkey="dynamical_velocity")
scv.tl.velocity_graph(adata, vkey="dynamical_velocity")

V_dyn = adata.layers["dynamical_velocity"]
if not isinstance(V_dyn, np.ndarray):
    V_dyn = V_dyn.toarray()

np.save(f"{outdir}/V_dynamical.npy", V_dyn)
print("Saved V_dynamical:", V_dyn.shape)

# --------------------------
# 8. Diffusion pseudotime (masked)
# --------------------------
adata.uns["iroot"] = 0

sc.tl.diffmap(adata)
sc.tl.dpt(adata)

pseudotime = adata.obs["dpt_pseudotime"].to_numpy()
pseudotime = (pseudotime - pseudotime.min()) / (
    pseudotime.max() - pseudotime.min() + 1e-8
)

pseudotime_masked = pseudotime.copy()
pseudotime_masked[~valid_pseudotime_mask] = np.nan

np.save(f"{outdir}/pseudotime.npy", pseudotime_masked)
print("Saved pseudotime:", pseudotime_masked.shape)

print("\n✅ Finished dentate gyrus preprocessing (gene-space layers saved).")

In [ ]:
import os
import numpy as np
import anndata as ad
import scvelo as scv
from sklearn.preprocessing import StandardScaler
import joblib
from scipy import sparse
import warnings
import flowmap
from flowmap import *

warnings.filterwarnings("ignore", category=FutureWarning)

# --------------------------
# 1. Paths / output
# --------------------------
input_h5ad = "./data/larry/larry_processed.h5ad"
input_embedder = "./data/larry/larry_flowmap_embedder.pkl"
outdir = "./data/benchmark/velocity_estimation/larry"
os.makedirs(outdir, exist_ok=True)

# --------------------------
# 2. Helpers
# --------------------------
def to_dense(x):
    if sparse.issparse(x):
        return x.toarray()
    return np.asarray(x)

def safe_layer(adata, key):
    if key not in adata.layers:
        raise ValueError(f"Missing required layer: '{key}'")
    return adata.layers[key]

def process_count_matrix(M, hvg_mask):
    """
    Match the original X-processing as closely as possible:
      - restrict to HVGs
      - log1p
      - zero-mean, unit-variance scaling
    """
    M = to_dense(M).copy()
    M = M[:, hvg_mask]
    M = np.log1p(M)
    scaler = StandardScaler(with_mean=True, with_std=True)
    M = scaler.fit_transform(M)
    return M

def process_velocity_matrix(V, hvg_mask):
    """
    Match the original V-processing as closely as possible:
      - restrict to HVGs
      - replace NaNs gene-wise
      - variance-scale only
    """
    V = to_dense(V).copy()
    V = V[:, hvg_mask]

    gene_means = np.nanmean(V, axis=0)
    gene_means[np.isnan(gene_means)] = 0.0
    inds = np.where(np.isnan(V))
    if len(inds[0]) > 0:
        V[inds] = np.take(gene_means, inds[1])

    scaler = StandardScaler(with_mean=False, with_std=True)
    V = scaler.fit_transform(V)
    return V

def get_hvg_mask(adata):
    if "highly_variable" in adata.var:
        hvg_mask = adata.var["highly_variable"].values.astype(bool)
        if hvg_mask.sum() == 0:
            raise ValueError("'highly_variable' exists but contains no True values.")
        print(f"Using existing HVG mask: {hvg_mask.sum()} genes")
        return hvg_mask

    print("No existing HVG mask found. Computing HVGs with scVelo filter_and_normalize.")
    scv.pp.filter_and_normalize(
        adata,
        min_shared_counts=20,
        n_top_genes=2000,
        log=False,
    )
    if "highly_variable" not in adata.var:
        raise ValueError("Failed to compute HVGs.")
    hvg_mask = adata.var["highly_variable"].values.astype(bool)
    print(f"Computed HVG mask: {hvg_mask.sum()} genes")
    return hvg_mask

# --------------------------
# 3. Load dataset
# --------------------------
adata_base = ad.read_h5ad(input_h5ad)
print("Loaded Larry dataset:", adata_base)

if "spliced" not in adata_base.layers or "unspliced" not in adata_base.layers:
    raise ValueError("Larry AnnData must contain 'spliced' and 'unspliced' layers.")

# --------------------------
# 4. Load FlowMap embedding
# --------------------------
emb = joblib.load(input_embedder)
print("Loaded FlowMap embedder:", type(emb))

if not hasattr(emb, "X_emb"):
    raise ValueError("Embedder does not have attribute 'X_emb'.")

X_emb = np.asarray(emb.X_emb)
print("Loaded FlowMap embedding:", X_emb.shape)

# --------------------------
# 5. scVelo preprocessing base
# --------------------------
# Work on a copy so the original object stays untouched.
adata_prep = adata_base.copy()

# Keep procedure as close as possible: use existing HVG mask if present.
hvg_mask = get_hvg_mask(adata_prep)

# scVelo moments are required for stochastic and dynamical velocity.
# If neighbors / moments already exist, this is usually cheap or reused.
scv.pp.moments(adata_prep, n_pcs=30, n_neighbors=30)
print("Computed scVelo moments.")

# --------------------------
# 6. Save shared matrices once
# --------------------------
# These are shared across modes.
spliced_mat = process_count_matrix(safe_layer(adata_prep, "spliced"), hvg_mask)
unspliced_mat = process_count_matrix(safe_layer(adata_prep, "unspliced"), hvg_mask)

print(
    f"Processed shared matrices:\n"
    f"  spliced   shape: {spliced_mat.shape}\n"
    f"  unspliced shape: {unspliced_mat.shape}\n"
    f"  embedding shape: {X_emb.shape}"
)

# --------------------------
# 7. scVelo velocity (vanilla, two modes)
# --------------------------

# ---- STOCHASTIC MODE ----
print("\nRunning scVelo (stochastic)...")

adata_stoch = adata_prep.copy()

scv.tl.velocity(adata_stoch, mode="stochastic", n_jobs=15)
scv.tl.velocity_graph(adata_stoch, n_jobs=15)

V_stoch = adata_stoch.layers["velocity"]
V_stoch = process_velocity_matrix(V_stoch, hvg_mask)

print(
    f"Stochastic velocity:\n"
    f"  shape: {V_stoch.shape}\n"
    f"  NaNs: {np.isnan(V_stoch).sum()}"
)

np.save(f"{outdir}/velocity_stochastic.npy", V_stoch)


# ---- DYNAMICAL MODE ----
print("\nRunning scVelo (dynamical)...")

adata_dyn = adata_prep.copy()

scv.tl.recover_dynamics(adata_dyn, n_jobs=15)
scv.tl.velocity(adata_dyn, mode="dynamical", n_jobs=15)
scv.tl.velocity_graph(adata_dyn, n_jobs=15)

V_dyn = adata_dyn.layers["velocity"]
V_dyn = process_velocity_matrix(V_dyn, hvg_mask)

print(
    f"Dynamical velocity:\n"
    f"  shape: {V_dyn.shape}\n"
    f"  NaNs: {np.isnan(V_dyn).sum()}"
)

np.save(f"{outdir}/velocity_dynamical.npy", V_dyn)


# --------------------------
# 8. Save shared matrices (once)
# --------------------------
np.save(f"{outdir}/spliced.npy", spliced_mat)
np.save(f"{outdir}/unspliced.npy", unspliced_mat)
np.save(f"{outdir}/embedding.npy", X_emb)

print(f"\nFinished processing Larry dataset. Saved to: {outdir}/")

In [ ]:
# --------------------------
# 9. Dumb visualization pseudotime (distance from ref)
# --------------------------
# Dumb reference point (given)
ref = np.array([3.0868282, 3.5019479])

# Euclidean distance from ref
distance_larry = np.linalg.norm(X_emb - ref[None, :], axis=1)

# Normalize to [0, 1] (just for nicer colors)
distance_larry = (distance_larry - distance_larry.min()) / (
    distance_larry.max() - distance_larry.min() + 1e-8
)

np.save(f"{outdir}/distance_pseudotime.npy", distance_larry)
print("Saved visualization pseudotime color:", distance_larry.shape)